# 00 — Setup (hypstab, from-scratch code)

Run this notebook first in Google Colab. It
1. gets the code from GitHub (branch `from-scratch`),
2. installs the few libraries Colab does not have,
3. checks whether a GPU is available,
4. runs the hypstab test suite,
5. loads the Cora config and starts a tiny run, to show what every run saves.

Runtime → Change runtime type → GPU is optional here (nothing heavy runs).

In [ ]:
# 1. Get the code (clone on first run, pull afterwards)
import os, subprocess
REPO_URL = "https://github.com/IritKogan/HypDiff_Network-Stability.git"
BRANCH = "from-scratch"
REPO_DIR = "/content/HypDiff_Network-Stability"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "50", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install what Colab lacks. Colab's own torch is kept (its version is recorded in run_info.json).
!pip install -q torch-geometric==2.8.0.post1 pytest

In [ ]:
# 3. GPU check
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 4. Test suite of the new package (CPU, ~10 s)
!python -m pytest -q -p no:warnings tests/hypstab_tests

In [ ]:
# 5. Load the config and start a tiny run: shows what is saved with every run
import json, sys
sys.path.insert(0, REPO_DIR)
from hypstab.config import load_config, config_to_toml
from hypstab.runinfo import start_run

config = load_config("configs/hypstab/cora.toml")
print(config_to_toml(config))
run_dir, device = start_run(config, run_dir="/content/hypstab_runs/setup_check")  # delete the folder to rerun
print("run dir:", run_dir, "| device:", device)
print(json.dumps(json.load(open(run_dir / "run_info.json")), indent=2))

### Quick look at the geometry (Stage 1)
A point is mapped into the Poincaré ball and back; its distance from the origin follows the closed form
`d(0, x) = 2·artanh(√c·‖x‖)/√c`.

In [ ]:
from hypstab.geometry import poincare as pb
c = config.geometry.curvature
v = torch.tensor([[0.3, -0.4], [2.0, 1.0]], dtype=torch.float64)
x = pb.expmap0(v, c)
print("points in the ball:", x, "\nnorms (< 1/sqrt(c)):", x.norm(dim=-1))
print("back to tangent space:", pb.logmap0(x, c))
print("d(0, x):", pb.distance(torch.zeros_like(x), x, c))
print("closed form:", 2 * torch.atanh(c**0.5 * x.norm(dim=-1)) / c**0.5)